# Notebook 4 : SVMs et noyaux

Notebook préparé par [Chloé-Agathe Azencott](http://cazencott.info) avec l'aide de Matthieu Najm.

Dans ce notebook il s'agit de découvrir les SVM (machines à vecteur de support) linéaires et à noyaux (non-linéaires).

In [ ]:
# charger numpy as np, matplotlib as plt
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
plt.rc('font', **{'size': 12}) # règle la taille de police globalement pour les plots (en pt)

In [ ]:
import pandas as pd

## 1. Chargement des données

Nous allons essayer de prédire l'espèce d'un manchot à partir de ses caractéristiques physiques à l'aide de Support Vector Machines (SVM).

Pour cela nous allons utiliser le dataset *palmerpenguins*, une alternative au desormais classique *iris* de scikit-learn.
Pour plus d'informations, vous pouvez allez voir le site: https://allisonhorst.github.io/palmerpenguins/index.html

Il contient les caractéristiques de trois espèces de manchots recensés sur l'archipel Palmer, au large de la côte nord-ouest de la peninsule Antarctique.

In [ ]:
palmerpenguins = pd.read_csv("data/penguins.csv")

__Alternativement :__ Si vous avez besoin de télécharger le fichier (par exemple sur colab) :

In [ ]:
!wget https://raw.githubusercontent.com/ThomasWalter/CourseFoundationsML/SIA2025/Notebooks/4-SVM/data/penguins.csv

palmerpenguins = pd.read_csv("penguins.csv")

In [ ]:
palmerpenguins.head()

### Description des données

In [ ]:
print(palmerpenguins.shape)

In [ ]:
import collections
print(collections.Counter(palmerpenguins.species))

**344** pingouins avec **8** attributs:
- *species* (l'espèce): Adelie (152 pingouins), Gentoo (124 pingouins) et Ginstrap (68 pingouis)
- *island* (l'île de recensement): Biscoe, Dream et Torgersen
- *bill_length_mm* : la longueur du bec en mm
- *bil_depth_mm*: largeur du bec en mm
- *flipper_length_mm*: largeur des palmes en mm
- *body_mass_g*: poids en g
- *sex*: male et female
- *year*: année du recensement

### Données manquantes

In [ ]:
palmerpenguins.isnull().sum()

Vous remarquez qu'il y a certaines observations pour lesquelles il manque des informations. C'est ce que l'on appelle des **données manquantes** (ou **missing values**).
Nous décidons ici d'ignorer ces observations.

In [ ]:
palmerpenguins = palmerpenguins[palmerpenguins['bill_depth_mm'].notna()]
palmerpenguins = palmerpenguins.reset_index()
palmerpenguins.shape

Nous n'avons donc plus que 342 échantillons.

Au lieu d'ignorer les observations incomplètes, une solution qui s'avère souvent meilleure (du point de vue des performances de modèles décisionnels construits à partir des données) consiste à **estimer** (ou **impute**) les données manquantes et à traiter les valeurs estimées comme des valeurs mesurées.

Pour cela nous aurions pu utiliser la fonction *SimpleImputer* de *sklearn.impute*. Pour plus d'informations, vous pouvez aller voir le site: https://scikit-learn.org/stable/modules/impute.html#impute

### Variables

Par la suite, nous allons seulement nous intéresser aux variables numériques *bill_depth_mm*,  *bill_length_mm*, *flipper_length_mm* et *body_mass_g*.

In [ ]:
penguins_features = palmerpenguins[["bill_length_mm", "bill_depth_mm","body_mass_g", "flipper_length_mm"]]

### Etiquettes

Nous allons essayer de prédire l'espèce, qui correspond à **species** mais sous forme d'entiers. Il sera alors plus facile de manier l'espèce sous forme d'entiers que de texte.

In [ ]:
species_names, species_int = np.unique(palmerpenguins.species, return_inverse=True)
print(species_names)

In [ ]:
penguins_labels = pd.DataFrame(palmerpenguins["species"])
penguins_labels["species_int"] = species_int

In [ ]:
penguins_labels

## 2. SVM linéaire (cas linéairement séparable)

Nous allons ici nous limiter à deux espèces : *Adelie (0)* et *Gentoo (2)* et deux variables : *bill_length_mm* et *bill_depth_mm*.

### Restriction des données aux deux variables et deux étiquettes choisies

In [ ]:
labels = penguins_labels[penguins_labels["species_int"].isin([0,2])]
labels = np.array(labels["species_int"])
print("shape de y:", labels.shape)

data = penguins_features[penguins_labels["species_int"].isin([0,2])]
data = np.array(data[["bill_depth_mm", "bill_length_mm"]])
print("shape de X:", data.shape)

### Visualisation des données

In [ ]:
plt.figure(figsize=(8, 6))

# afficher les données (pingouins Adelie)
adelie_indices = np.where(labels==0)[0]
adelie = plt.scatter(data[adelie_indices][:, 0],
                    data[adelie_indices][:, 1],
                    label = 'Adelie',
                    cmap=plt.cm.Paired)

# afficher les données (pingouins Gentoo)
gentoo_indices = np.where(labels==2)[0]
gentoo = plt.scatter(data[gentoo_indices][:, 0],
                    data[gentoo_indices][:, 1],
                    label = 'Gentoo',
                    cmap=plt.cm.Paired)

# Légende
plt.legend()
plt.xlabel("Bill depth (mm)")
plt.ylabel("Bill length (mm)")
plt.title("Données")
plt.tight_layout()
plt.show()

__Question :__ Le problème de classification vous parait-il facile ou difficile ? Pourrez-vous entrainer un modèle linéaire ?

**Réponse :**

Le problème de classification **semble relativement facile**.

En observant la visualisation, les deux classes (Adelie et Gentoo) semblent être **bien séparées** dans l'espace bidimensionnel formé par la profondeur du bec (Bill depth) et la longueur du bec (Bill length).

**Oui, on peut entraîner un modèle linéaire.**

Les deux groupes de pingouins présentent une **séparabilité linéaire claire** - il est possible de tracer une ligne droite (ou un hyperplan en général) qui sépare les deux classes avec peu ou pas d'erreurs de classification. Cela signifie qu'un classifieur linéaire (comme une régression logistique ou un SVM linéaire) devrait fonctionner très bien sur cet ensemble de données.

Les modèles linéaires sont également plus simples, plus rapides à entraîner et généralement plus faciles à interpréter, ce qui les rend préférables lorsque les données sont linéairement séparables comme c'est le cas ici.

### SVM linéaire

Nous allons utiliser la classe [SVC](http://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html#sklearn.svm.SVC) du module `svm` de scikit-learn pour entraîner une SVM linéaire sur ces données.

In [ ]:
from sklearn import svm

In [ ]:
# initialisation
model_svc = svm.SVC(kernel='linear', C=10)

# entrainement
model_svc.fit(data, labels)

Affichons la performance du prédicteur :

In [ ]:
print("Score de la SVM linéaire (C=10): %.2f" % model_svc.score(data, labels))

__Question :__ De quel score s'agit-il ? Utilisez
```
help(model_svc.score)
```

In [ ]:
help(model_svc.score)

__Question :__ Que signifie cette performance ?

**Réponse :**

Un score de **1.00 (soit 100%)** signifie que :

Interprétation :
- La méthode `score()` pour un modèle SVM retourne la **précision globale** (accuracy) du modèle sur les données fournies.
- Avec un score de 1.00, cela signifie que **le modèle SVM linéaire classifie correctement 100% des exemples** (ou quasiment tous) dans l'ensemble de données.

Implications :
1. **Pas d'erreur de classification** : Le modèle entraîné ne fait aucune erreur de prédiction sur ces données.
2. **Séparation linéaire parfaite** : Cela confirme que les deux classes (Adelie et Gentoo) sont **linéairement séparables** comme nous l'avions observé dans la visualisation.
3. **Données d'entraînement** : Important à noter que ce score est calculé sur les mêmes données utilisées pour entraîner le modèle (pas de séparation train/test), donc il ne reflète pas nécessairement la performance générale du modèle sur de nouvelles données.

**Remarque importante**
Pour évaluer la vraie performance du modèle, il faudrait utiliser une **validation croisée** ou une **séparation train/test** pour éviter le surapprentissage apparent et obtenir une estimation plus fiable de la généralisation du modèle.

Nous pouvons aussi afficher la matrice de confusion :

In [ ]:
from sklearn import metrics

In [ ]:
metrics.ConfusionMatrixDisplay.from_predictions(labels, model_svc.predict(data))

Alternativement :

In [ ]:
metrics.confusion_matrix(labels, model_svc.predict(data))

### Hyperplan séparateur

Nous pouvons maintenant visualiser la frontière de décision :

In [ ]:
plt.figure(figsize=(8, 6))

# afficher les données (pingouins Adelie)
adelie_indices = np.where(labels==0)[0]
adelie = plt.scatter(data[adelie_indices][:, 0],
                    data[adelie_indices][:, 1],
                    label = 'Adelie',
                    cmap=plt.cm.Paired)

# afficher les données (pingouins Gentoo)
gentoo_indices = np.where(labels==2)[0]
gentoo = plt.scatter(data[gentoo_indices][:, 0],
                    data[gentoo_indices][:, 1],
                    label = 'Gentoo',
                    cmap=plt.cm.Paired)

# Limites du cadre
ax = plt.gca()
xlim = ax.get_xlim()
ylim = ax.get_ylim()

# Marquer les vecteurs de support d'une croix
ax.scatter(model_svc.support_vectors_[:, 0],
           model_svc.support_vectors_[:, 1],
           linewidth=1,
           marker='x', s=200,
           color='k')

# Grille de points sur lesquels appliquer le modèle
xx = np.linspace(xlim[0], xlim[1], 30)
yy = np.linspace(ylim[0], ylim[1], 30)
YY, XX = np.meshgrid(yy, xx)
xy = np.vstack([XX.ravel(), YY.ravel()]).T
# Prédire pour les points de la grille
Z = model_svc.decision_function(xy).reshape(XX.shape)

# Afficher la frontière de décision et la marge
ax.contour(XX, YY, Z, colors='k', levels=[-1, 0, 1],
           alpha=0.5, linestyles=['--', '-', '--'])

# Légende
plt.legend()
plt.xlabel("Bill depth (mm)")
plt.ylabel("Bill length (mm)")
plt.title("SVM linéaire (C=10)")
plt.tight_layout()
plt.show()

__Question :__ Quels points sont vecteurs de support ?

**Réponse :**

Les vecteurs de support sont les points de données qui sont situés **sur la marge** ou **à l'intérieur de la marge** (en cas de classes non parfaitement séparables). Ce sont ces points qui déterminent la position de l'hyperplan de séparation et la largeur de la marge.

Dans le graphique précédent, les vecteurs de support sont clairement identifiés par les **croix noires ('x')** qui ont été ajoutées sur le plot. On peut voir qu'ils sont situés sur les lignes en pointillé qui définissent la marge.

### Avec C plus petit

__Question :__ À quoi peut-on s'attendre avec une valeur de C plus faible ?

**Réponse :**

Le paramètre `C` dans une SVM contrôle la pénalité pour les erreurs de classification. Plus `C` est petit, plus la pénalité pour une erreur de classification est faible. Cela signifie que le modèle va privilégier une marge plus grande, même si cela entraîne plus d'erreurs de classification sur les données d'entraînement.

Donc, avec une valeur de `C` plus faible, on peut s'attendre à :

1.  **Une marge de séparation plus large :** Le modèle sera plus tolérant aux points mal classés (outliers ou points chevauchant les classes) et cherchera à maximiser la distance entre l'hyperplan de séparation et les points de données les plus proches des deux classes.
2.  **Un nombre potentiellement plus élevé de vecteurs de support :** Plus de points de données pourraient se retrouver sur ou à l'intérieur de la marge.
3.  **Moins de surapprentissage :** Une marge plus large conduit généralement à un modèle plus généralisable, moins sensible aux bruits et aux spécificités des données d'entraînement. C'est un moyen de régularisation.
4.  **Une précision sur les données d'entraînement potentiellement plus faible :** En étant plus tolérant aux erreurs, le modèle pourrait avoir un score d'exactitude (accuracy) légèrement inférieur sur les données d'entraînement, mais potentiellement un meilleur score de généralisation sur des données de test non vues.

Vérifions cela en pratique :

In [ ]:
# initialisation
model_svc_01 = svm.SVC(kernel='linear', C=0.1)

# entrainement
model_svc_01.fit(data, labels)

Affichons la performance du prédicteur :

In [ ]:
print("Score de la SVM linéaire (C=0.1): %.2f" % model_svc_01.score(data, labels))

In [ ]:
metrics.ConfusionMatrixDisplay.from_predictions(labels, model_svc_01.predict(data))

Alternativement :

In [ ]:
metrics.confusion_matrix(labels, model_svc_01.predict(data))

Visualisons la nouvelle frontière de décision :

In [ ]:
plt.figure(figsize=(8, 6))

# afficher les données (pingouins Adelie)
adelie_indices = np.where(labels==0)[0]
adelie = plt.scatter(data[adelie_indices][:, 0],
                    data[adelie_indices][:, 1],
                    label = 'Adelie',
                    cmap=plt.cm.Paired)

# afficher les données (pingouins Gentoo)
gentoo_indices = np.where(labels==2)[0]
gentoo = plt.scatter(data[gentoo_indices][:, 0],
                    data[gentoo_indices][:, 1],
                    label = 'Gentoo',
                    cmap=plt.cm.Paired)

# Limites du cadre
ax = plt.gca()
xlim = ax.get_xlim()
ylim = ax.get_ylim()

## Modèle précédent
# Marquer les vecteurs de support d'une croix
#ax.scatter(model_svc.support_vectors_[:, 0],
#           model_svc.support_vectors_[:, 1],
#           linewidth=1,
#           marker='x', s=200,
#           color='k')

# Grille de points sur lesquels appliquer le modèle
xx = np.linspace(xlim[0], xlim[1], 30)
yy = np.linspace(ylim[0], ylim[1], 30)
YY, XX = np.meshgrid(yy, xx)
xy = np.vstack([XX.ravel(), YY.ravel()]).T
# Prédire pour les points de la grille
Z = model_svc.decision_function(xy).reshape(XX.shape)

# Afficher la frontière de décision et la marge
#ax.contour(XX, YY, Z, colors='k', levels=[-1, 0, 1],
#           alpha=0.5, linestyles=['--', '-', '--'])

## Nouveau modèle
# Marquer les vecteurs de support d'une croix
ax.scatter(model_svc_01.support_vectors_[:, 0],
           model_svc_01.support_vectors_[:, 1],
           linewidth=1,
           marker='+', s=200,
           color='k')

# Grille de points sur lesquels appliquer le modèle
xx = np.linspace(xlim[0], xlim[1], 30)
yy = np.linspace(ylim[0], ylim[1], 30)
YY, XX = np.meshgrid(yy, xx)
xy = np.vstack([XX.ravel(), YY.ravel()]).T
# Prédire pour les points de la grille
Z = model_svc_01.decision_function(xy).reshape(XX.shape)

# Afficher la frontière de décision et la marge
ax.contour(XX, YY, Z, colors='g', levels=[-1, 0, 1],
           alpha=0.5, linestyles=['--', '-', '--'])

# Légende
plt.legend()
plt.xlabel("Bill depth (mm)")
plt.ylabel("Bill length (mm)")
plt.title("SVM linéaire (C=0.1)")
plt.tight_layout()
plt.show()

## 3. SVM linéaire (cas non-linéairement séparable)

Utilisons maintenant les deux variables *body_mass_g* et *bill_length_mm*.

### Restriction des données aux deux variables et deux étiquettes choisies

In [ ]:
labels = penguins_labels[penguins_labels["species_int"].isin([0,2])]
labels = np.array(labels["species_int"])
print("shape de y:", labels.shape)

data = penguins_features[penguins_labels["species_int"].isin([0,2])]
data = np.array(data[["body_mass_g", "bill_length_mm"]])
print("shape de X:", data.shape)

### Visualisation des données

In [ ]:
plt.figure(figsize=(8, 6))

# afficher les données (pingouins Adelie)
adelie_indices = np.where(labels==0)[0]
adelie = plt.scatter(data[adelie_indices][:, 0],
                    data[adelie_indices][:, 1],
                    label = 'Adelie',
                    cmap=plt.cm.Paired)

# afficher les données (pingouins Gentoo)
gentoo_indices = np.where(labels==2)[0]
gentoo = plt.scatter(data[gentoo_indices][:, 0],
                    data[gentoo_indices][:, 1],
                    label = 'Gentoo',
                    cmap=plt.cm.Paired)

# Légende
plt.legend()
plt.xlabel("Body mass (g)")
plt.ylabel("Bill length (mm)")
plt.title("Données")
plt.tight_layout()
plt.show()

__Question :__ Le problème de classification vous parait-il facile ou difficile ? Pourrez-vous entrainer un modèle linéaire ?

**Réponse :**

En observant le nuage de points pour les variables `body_mass_g` et `bill_length_mm`, le problème de classification semble **plus difficile** que le cas précédent (`bill_depth_mm` et `bill_length_mm`).

Les deux espèces, Adelie et Gentoo, **ne semblent pas être linéairement séparables de manière parfaite**. Il y a un chevauchement notable entre les deux groupes de points, ce qui signifie qu'il ne sera pas possible de tracer une simple ligne droite qui sépare complètement les deux classes sans erreurs.

**Oui, il est toujours possible d'entraîner un modèle linéaire**, mais on peut s'attendre à ce que la performance (l'accuracy) soit inférieure à 100% sur les données d'entraînement. Le modèle linéaire tentera de trouver le meilleur hyperplan de séparation qui minimise les erreurs, même s'il ne peut pas les éliminer complètement. C'est un cas où un SVM linéaire avec une pénalité `C` (qui gère les erreurs de classification) sera pertinent.

__Question :__ Que pensez-vous des échelles prises par les deux variables ?

**Réponse :**

En observant le graphique précédent, on remarque une très grande différence dans les échelles des deux variables :

*   `body_mass_g` (masse corporelle) se situe dans une plage allant de ~2700g à ~6000g.
*   `bill_length_mm` (longueur du bec) se situe dans une plage beaucoup plus petite, allant de ~30mm à ~60mm.

Cette disparité d'échelles est problématique pour de nombreux algorithmes d'apprentissage automatique, y compris les SVM. Un algorithme basé sur la distance (comme la SVM, qui utilise la distance euclidienne dans son calcul de marge) sera fortement influencé par la variable ayant la plus grande plage de valeurs. Dans ce cas, `body_mass_g` dominerait les calculs de distance, et les variations de `bill_length_mm` seraient moins prises en compte.

C'est pourquoi il est crucial de **centrer et réduire** (standardiser) les données pour que toutes les caractéristiques contribuent de manière équitable à la mesure de distance, et pour éviter que l'algorithme ne soit biaisé par des variables aux grandes échelles.

### Transformation des variables

Nous allons maintenant centrer-réduire les données.

In [ ]:
from sklearn import preprocessing

In [ ]:
# standardisation (centrer-réduire)
std_scaler = preprocessing.StandardScaler().fit(data)
data_scaled = std_scaler.transform(data)

In [ ]:
plt.figure(figsize=(8, 6))

# afficher les données (pingouins Adelie)
adelie_indices = np.where(labels==0)[0]
adelie = plt.scatter(data_scaled[adelie_indices][:, 0],
                    data_scaled[adelie_indices][:, 1],
                    label = 'Adelie',
                    cmap=plt.cm.Paired)

# afficher les données (pingouins Gentoo)
gentoo_indices = np.where(labels==2)[0]
gentoo = plt.scatter(data_scaled[gentoo_indices][:, 0],
                    data_scaled[gentoo_indices][:, 1],
                    label = 'Gentoo',
                    cmap=plt.cm.Paired)

# Légende
plt.legend()
plt.xlabel("Body mass (centrée-réduite)")
plt.ylabel("Bill length (centrée-réduite)")
plt.title("Données centrées-réduites")
plt.tight_layout()
plt.show()

### SVM linéaire

In [ ]:
# initialisation
model_svc = svm.SVC(kernel='linear', C=10)

# entrainement
model_svc.fit(data_scaled, labels)

Affichons la performance du prédicteur :

In [ ]:
print("Score de la SVM linéaire (C=10): %.2f" % model_svc.score(data_scaled, labels))

In [ ]:
metrics.ConfusionMatrixDisplay.from_predictions(labels, model_svc.predict(data_scaled))

Alternativement :

In [ ]:
metrics.confusion_matrix(labels, model_svc.predict(data_scaled))

__Question :__ Cette performance correspond-elle à vos attentes ?

**Réponse :**

Oui, cette performance de **0.96 (soit 96%)** correspond bien à nos attentes.

Comme nous l'avons anticipé lors de la visualisation des données centrées-réduites, les deux classes (Adelie et Gentoo) ne sont pas parfaitement linéairement séparables avec les variables `body_mass_g` et `bill_length_mm`. Il y avait un chevauchement notable entre les groupes de points.

Un score de 1.00 n'était donc pas attendu pour un SVM linéaire. Une précision de 96% est un excellent résultat compte tenu de la complexité du problème de séparation observé. La matrice de confusion confirme qu'un petit nombre de points sont mal classés, ce qui est cohérent avec la non-séparabilité linéaire parfaite.

### Hyperplan séparateur

Nous pouvons maintenant visualiser la frontière de décision :

In [ ]:
plt.figure(figsize=(8, 6))

# afficher les données (pingouins Adelie)
adelie_indices = np.where(labels==0)[0]
adelie = plt.scatter(data_scaled[adelie_indices][:, 0],
                    data_scaled[adelie_indices][:, 1],
                    label = 'Adelie',
                    cmap=plt.cm.Paired)

# afficher les données (pingouins Gentoo)
gentoo_indices = np.where(labels==2)[0]
gentoo = plt.scatter(data_scaled[gentoo_indices][:, 0],
                    data_scaled[gentoo_indices][:, 1],
                    label = 'Gentoo',
                    cmap=plt.cm.Paired)


# Limites du cadre
ax = plt.gca()
xlim = ax.get_xlim()
ylim = ax.get_ylim()

# Marquer les vecteurs de support d'une croix
ax.scatter(model_svc.support_vectors_[:, 0],
           model_svc.support_vectors_[:, 1],
           linewidth=1,
           marker='x',
           color='k')

# Grille de points sur lesquels appliquer le modèle
xx = np.linspace(xlim[0], xlim[1], 30)
yy = np.linspace(ylim[0], ylim[1], 30)
YY, XX = np.meshgrid(yy, xx)
xy = np.vstack([XX.ravel(), YY.ravel()]).T
# Prédire pour les points de la grille
Z = model_svc.decision_function(xy).reshape(XX.shape)

# Afficher la frontière de décision et la marge
ax.contour(XX, YY, Z, colors='k', levels=[-1, 0, 1],
           alpha=0.5, linestyles=['--', '-', '--'])

# Légende
plt.legend()
plt.xlabel("Body mass (centrée-réduite)")
plt.ylabel("Bill length (centrée-réduite)")
plt.title("Frontière de décision de la SVM linéaire (C=10)")
plt.tight_layout()
plt.show()

__Question :__ Quels points sont vecteurs de support ?

**Réponse :**

Dans le contexte de ce graphique où les classes ne sont pas parfaitement linéairement séparables (ce qui est géré par le paramètre `C`), les vecteurs de support sont les points de données qui se trouvent **sur la marge** ou **entre les marges et l'hyperplan de séparation**, ainsi que les points **mal classés** qui tombent du mauvais côté de la marge ou de l'hyperplan.

Visuellement, dans le graphique précédent, les vecteurs de support sont toujours les points identifiés par les **croix noires ('x')**. On peut observer qu'ils sont situés le long des lignes en pointillé (les marges) ou parfois à l'intérieur de ces marges (les points mal classés ou ceux qui "pénètrent" un peu dans la région de l'autre classe en raison de la tolérance introduite par `C`). Ces points sont cruciaux car ils définissent l'hyperplan de séparation et la largeur de la marge.

### Avec C plus petit

__Question :__ À quoi peut-on s'attendre avec une valeur de C plus faible ?

**Réponse :**

Comme précédemment, le paramètre `C` dans une SVM contrôle la pénalité pour les erreurs de classification. Plus `C` est petit, plus la pénalité pour une erreur de classification est faible. Cela signifie que le modèle va privilégier une marge plus grande, même si cela entraîne plus d'erreurs de classification sur les données d'entraînement.

Donc, avec une valeur de `C` plus faible, on peut s'attendre à :

1.  **Une marge de séparation plus large :** Le modèle sera plus tolérant aux points mal classés (outliers ou points chevauchant les classes) et cherchera à maximiser la distance entre l'hyperplan de séparation et les points de données les plus proches des deux classes.
2.  **Un nombre potentiellement plus élevé de vecteurs de support :** Plus de points de données pourraient se retrouver sur ou à l'intérieur de la marge.
3.  **Moins de surapprentissage :** Une marge plus large conduit généralement à un modèle plus généralisable, moins sensible aux bruits et aux spécificités des données d'entraînement. C'est un moyen de régularisation.
4.  **Une précision sur les données d'entraînement potentiellement plus faible :** En étant plus tolérant aux erreurs, le modèle pourrait avoir un score d'exactitude (accuracy) légèrement inférieur sur les données d'entraînement, mais potentiellement un meilleur score de généralisation sur des données de test non vues.

Vérifions cela en pratique :

In [ ]:
# initialisation
model_svc_01 = svm.SVC(kernel='linear', C=0.01)

# entrainement
model_svc_01.fit(data_scaled, labels)

In [ ]:
print("Score de la SVM linéaire (données centrées-réduites, C=0.1): %.2f" % model_svc_01.score(data_scaled, labels))

In [ ]:
metrics.ConfusionMatrixDisplay.from_predictions(labels, model_svc_01.predict(data_scaled))

Alternativement :

In [ ]:
metrics.confusion_matrix(labels, model_svc_01.predict(data_scaled))

__Question :__ Comment la performance a-t-elle évolué ?

**Réponse :**

En comparant la performance du modèle avec `C=0.01` à celle du modèle avec `C=10` :

*   **Score d'accuracy :** Le score d'accuracy sur les données d'entraînement est resté le même, à **0.96** pour `C=0.01` (contre 0.96 pour `C=10`).

*   **Matrice de confusion :** Bien que l'accuracy globale n'ait pas changé, la répartition des erreurs a légèrement évolué :
    *   Avec `C=10`, nous avions 7 Adelie classés à tort comme Gentoo et 3 Gentoo classés à tort comme Adelie. (Total 10 erreurs)
    *   Avec `C=0.01`, nous avons 6 Adelie classés à tort comme Gentoo et 4 Gentoo classés à tort comme Adelie. (Total 10 erreurs)

La précision globale sur l'ensemble d'entraînement est donc identique. Cela signifie que même avec une valeur de `C` plus faible (qui favorise une marge plus large et une plus grande tolérance aux erreurs), le modèle parvient à classifier le même nombre total de points correctement sur cet ensemble de données. La faible modification dans la matrice de confusion indique simplement que les quelques points mal classés ont légèrement changé de catégorie, mais le nombre total reste le même.

Nous pouvons maintenant visualiser la frontière de décision :

In [ ]:
plt.figure(figsize=(8, 6))

# afficher les données (pingouins Adelie)
adelie = plt.scatter(data_scaled[np.where(penguins_labels["species_int"]==0),0],
                    data_scaled[np.where(penguins_labels["species_int"]==0),1],
                    s=50,
                    label = 'Adelie',
                    cmap=plt.cm.Paired)

# afficher les données (pingouins Gentoo)
gentoo = plt.scatter(data_scaled[np.where(penguins_labels["species_int"]==2),0],
                    data_scaled[np.where(penguins_labels["species_int"]==2),1],
                    label = 'Gentoo',
                    cmap=plt.cm.Paired)

# Limites du cadre
ax = plt.gca()
xlim = ax.get_xlim()
ylim = ax.get_ylim()

# Marquer les vecteurs de support d'une croix
ax.scatter(model_svc_01.support_vectors_[:, 0],
           model_svc_01.support_vectors_[:, 1],
           linewidth=1,
           marker='x',
           color='k')

# Grille de points sur lesquels appliquer le modèle
xx = np.linspace(xlim[0], xlim[1], 30)
yy = np.linspace(ylim[0], ylim[1], 30)
YY, XX = np.meshgrid(yy, xx)
xy = np.vstack([XX.ravel(), YY.ravel()]).T
# Prédire pour les points de la grille
Z = model_svc_01.decision_function(xy).reshape(XX.shape)

# Afficher la frontière de décision et la marge
ax.contour(XX, YY, Z, colors='k', levels=[-1, 0, 1],
           alpha=0.5, linestyles=['--', '-', '--'])

# Frontière précédente
# Grille de points sur lesquels appliquer le modèle
xx = np.linspace(xlim[0], xlim[1], 30)
yy = np.linspace(ylim[0], ylim[1], 30)
YY, XX = np.meshgrid(yy, xx)
xy = np.vstack([XX.ravel(), YY.ravel()]).T
# Prédire pour les points de la grille
Z = model_svc.decision_function(xy).reshape(XX.shape)

# Afficher la frontière de décision et la marge
ax.contour(XX, YY, Z, colors='g', levels=[-1, 0, 1],
           alpha=0.5, linestyles=['--', '-', '--'])


# Légende
plt.legend()
plt.xlabel("Body mass (g)")
plt.ylabel("Bill length (mm)")
plt.title("Frontière de décision de la SVM linéaire (C=0.01)")
plt.tight_layout()
plt.show()

## 4. Matrice de Gram

On peut interpréter le **noyau** (ou **kernel** ou **matrice de Gram**) comme une matrice de similarité entre les différentes observations. On s'appuie alors sur la ressemblance de certaines observations pour pouvoir les classifier.

Nous allons représenter la matrice de Gram associée au précédent classifieur. Dans le cas d'un SVM à noyau linéaire, il s'agit du produit scalaire des variables. Pour que vous puissiez intuiter de manière juste la similarité entre les observations, nous allons nous ramener à une matrice avec des 1 sur la diagonale grâce à la fonction *center_an_normalise_kernel()*.

In [ ]:
def center_and_normalise_kernel(K_temp):
    K_temp = preprocessing.KernelCenterer().fit_transform(K_temp)
    nb_item = K_temp.shape[0]
    K_norm = np.zeros((nb_item, nb_item))
    for i in range(nb_item):
        for j in range(i, nb_item):
            K_norm[i, j] = K_temp[i, j] / np.sqrt(K_temp[i, i] * K_temp[j, j])
            K_norm[j, i] = K_norm[i, j]

    return K_norm

In [ ]:
GramMatrix = np.inner(data_scaled, data_scaled)
GramMatrix_scaled = center_and_normalise_kernel(GramMatrix)

# heatmap + color map
fig, ax = plt.subplots(figsize=(5, 5))
plot = ax.imshow(GramMatrix_scaled)

# set axes boundaries
ax.set_xlim([0, data.shape[0]]) ; ax.set_ylim([0, data_scaled.shape[0]])

# flip the y-axis
ax.invert_yaxis() ; ax.xaxis.tick_top()

# plot colorbar to the right
plt.colorbar(plot, pad=0.1, fraction=0.04)
plt.show()

__Question:__ Que remarquez-vous ? Est-ce vous auriez pu anticiper le fait que le classifieur sépare bien en ne regardant que la matrice de Gram ? Remarquez que les observations sont ordonnées par étiquettes (d'abord les manchots Adelie puis les Gentoo).

**Réponse :**

En observant la matrice de Gram (la `heatmap`), on remarque une structure très distincte : deux blocs principaux de forte similarité (valeurs proches de 1, couleurs chaudes (jaune)) le long de la diagonale, et deux blocs de faible similarité (valeurs proches de -1 ou 0, couleurs froides/neutres (violet)) en dehors de ces blocs diagonaux.

Plus précisément :

*   Les observations des **manchots Adelie** (première partie des données) montrent une forte similarité entre elles (le premier bloc en haut à gauche est très clair/chaud).
*   Les observations des **manchots Gentoo** (deuxième partie des données) montrent également une forte similarité entre elles (le deuxième bloc en bas à droite est très clair/chaud).
*   En revanche, la similarité entre les manchots Adelie et les manchots Gentoo est faible (les blocs hors diagonale, en haut à droite et en bas à gauche, sont plus sombres/froids).

**Oui, on aurait pu anticiper que le classifieur séparerait bien les classes** en ne regardant que la matrice de Gram. La structure "en blocs" de la matrice, où les observations d'une même classe sont très similaires entre elles et peu similaires aux observations de l'autre classe, est une indication très forte d'une bonne séparabilité entre les classes. Un classifieur linéaire s'appuie sur cette similarité pour trouver une frontière. Une matrice de Gram avec des blocs diagonaux très clairs et des blocs hors diagonale très sombres (pour les classes différentes) suggère que les classes sont bien regroupées et distinctes, facilitant ainsi la tâche de séparation pour le SVM.

## 5. SVM linéaire (cas plus difficile)

Considérons maintenant les deux espèces *Adelie (0)* et *Chinstrap (1)* et les variables *body_mass_g* et *bill_depth_mm*.

### Restriction des données aux deux variables et deux étiquettes choisies

In [ ]:
labels = penguins_labels[penguins_labels["species_int"].isin([0,1])]
labels = np.array(labels["species_int"])
print("shape de y:", labels.shape)

data = penguins_features[penguins_labels["species_int"].isin([0,1])]
data = np.array(data[["body_mass_g", "bill_depth_mm"]])
print("shape de X:", data.shape)

### Transformation des variables

Nous allons maintenant centrer-réduire les données.

In [ ]:
# standardisation (centrer-réduire)
std_scaler = preprocessing.StandardScaler().fit(data)
data_scaled = std_scaler.transform(data)

### Visualisation des données

In [ ]:
plt.figure(figsize=(8, 6))

# afficher les données (pingouins Adelie)
adelie_indices = np.where(labels==0)[0]
adelie = plt.scatter(data_scaled[adelie_indices][:, 0],
                    data_scaled[adelie_indices][:, 1],
                    label = 'Adelie',
                    cmap=plt.cm.Paired)

# afficher les données (pingouins Chinstrap)
gentoo_indices = np.where(labels==1)[0]
gentoo = plt.scatter(data_scaled[gentoo_indices][:, 0],
                    data_scaled[gentoo_indices][:, 1],
                    label = 'Chinstrap',
                    cmap=plt.cm.Paired)

# Légende
plt.legend()
plt.xlabel("Body mass (centrée-réduite)")
plt.ylabel("Bill depth (centrée-réduite)")
plt.title("Données centrées-réduites")
plt.tight_layout()
plt.show()

__Question :__ Le problème de classification vous parait-il facile ou difficile ? Pourrez-vous entrainer un modèle linéaire ?

**Réponse :**

En observant la visualisation des données centrées-réduites pour les espèces Adelie et Chinstrap avec les variables `body_mass_g` et `bill_depth_mm`, le problème de classification semble **très difficile** pour un modèle linéaire.

Les deux classes présentent un **chevauchement important**, ce qui indique qu'il est **impossible de tracer une ligne droite (un hyperplan) qui puisse séparer correctement les deux espèces** sans commettre un grand nombre d'erreurs de classification.

**On peut toujours entraîner un modèle linéaire**, mais il est très probable que sa performance soit **faible**. Le modèle linéaire ne parviendra pas à capturer la complexité de la séparation entre ces deux classes, et on s'attendrait à une précision bien inférieure à ce que nous avons vu dans les cas précédents. Cela suggère qu'un modèle non-linéaire serait plus approprié pour ce scénario.

### SVM linéaire

In [ ]:
# initialisation
model_svc = svm.SVC(kernel='linear', C=10)

# entrainement
model_svc.fit(data_scaled, labels)

Affichons la performance du prédicteur :

In [ ]:
print("Score de la SVM linéaire (C=10): %.2f" % model_svc.score(data_scaled, labels))

__Question :__ Cette performance correspond-elle à vos attentes ?

**Réponse :**

Oui, cette performance de **0.69 (soit 69%)** correspond tout à fait à nos attentes.

Comme nous l'avions anticipé lors de la visualisation des données et dans la réponse à la question précédente, le problème de classification entre les espèces Adelie et Chinstrap avec ces deux variables (`body_mass_g` et `bill_depth_mm`) est **très difficile** pour un modèle linéaire en raison du chevauchement important entre les classes.

Un score d'exactitude (accuracy) de 69% sur les données d'entraînement pour un SVM linéaire est bas, mais il est cohérent avec l'observation que les classes ne sont pas linéairement séparables. Cela confirme la difficulté du problème pour ce type de modèle.

In [ ]:
metrics.ConfusionMatrixDisplay.from_predictions(labels, model_svc.predict(data_scaled))

Alternativement :

In [ ]:
metrics.confusion_matrix(labels, model_svc.predict(data_scaled))

__Question :__ Qu'a-t-on vraiment appris ici ?

**Réponse :**

En analysant la performance et la matrice de confusion (`array([[151, 0], [68, 0]])`), nous avons appris un point crucial :

Le modèle SVM linéaire (avec C=10) a prédit correctement tous les 151 manchots Adelie (classe 0). Cependant, il a **classifié les 68 manchots Chinstrap (classe 1) de manière complètement erronée, les attribuant tous à la classe Adelie (classe 0)**. Cela se traduit par 0 prédiction correcte pour la classe Chinstrap.

En d'autres termes, pour ces deux variables (`body_mass_g` et `bill_depth_mm`), le modèle linéaire n'a pas réussi à trouver une frontière de décision pertinente pour séparer les espèces Adelie et Chinstrap. Il a essentiellement appris à prédire systématiquement la classe Adelie, ce qui est une forme d'échec pour la classification binaire. Ce résultat met en évidence les **limites des modèles linéaires** lorsque les classes ne sont absolument pas linéairement séparables, comme l'indiquait la visualisation initiale des données.

Nous pouvons maintenant visualiser la frontière de décision :

In [ ]:
plt.figure(figsize=(8, 6))

# afficher les données (pingouins Adelie)
adelie_indices = np.where(labels==0)[0]
adelie = plt.scatter(data_scaled[adelie_indices][:, 0],
                    data_scaled[adelie_indices][:, 1],
                    label = 'Adelie',
                    cmap=plt.cm.Paired)

# afficher les données (pingouins Chinstrap)
gentoo_indices = np.where(labels==1)[0]
gentoo = plt.scatter(data_scaled[gentoo_indices][:, 0],
                    data_scaled[gentoo_indices][:, 1],
                    label = 'Chinstrap',
                    cmap=plt.cm.Paired)


# Limites du cadre
ax = plt.gca()
xlim = ax.get_xlim()
ylim = ax.get_ylim()

# Marquer les vecteurs de support d'une croix
ax.scatter(model_svc.support_vectors_[:, 0],
           model_svc.support_vectors_[:, 1],
           linewidth=1,
           marker='x',
           color='k')

# Grille de points sur lesquels appliquer le modèle
xx = np.linspace(xlim[0], xlim[1], 30)
yy = np.linspace(ylim[0], ylim[1], 30)
YY, XX = np.meshgrid(yy, xx)
xy = np.vstack([XX.ravel(), YY.ravel()]).T
# Prédire pour les points de la grille
Z = model_svc.decision_function(xy).reshape(XX.shape)

# Afficher la frontière de décision et la marge
ax.contour(XX, YY, Z, colors='k', levels=[-1, 0, 1],
           alpha=0.5, linestyles=['--', '-', '--'])

# Légende
plt.legend()
plt.xlabel("Body mass (centrée-réduite)")
plt.ylabel("Bill depth (centrée-réduite)")
plt.title("Frontière de décision de la SVM linéaire (C=10)")
plt.tight_layout()
plt.show()

### Matrice de Gram

In [ ]:
GramMatrix = np.inner(data_scaled, data_scaled)
GramMatrix_scaled = center_and_normalise_kernel(GramMatrix)

# heatmap + color map
fig, ax = plt.subplots(figsize=(5, 5))
plot = ax.imshow(GramMatrix_scaled)

# set axes boundaries
ax.set_xlim([0, data.shape[0]]) ; ax.set_ylim([0, data_scaled.shape[0]])

# flip the y-axis
ax.invert_yaxis() ; ax.xaxis.tick_top()

# plot colorbar to the right
plt.colorbar(plot, pad=0.1, fraction=0.04)
plt.show()

__Question :__ Qu'observez-vous maintenant ?

**Réponse :**

Contrairement à la matrice de Gram observée pour le cas linéairement séparable (Adelie vs Gentoo avec `body_mass_g` et `bill_length_mm`), cette nouvelle matrice de Gram (pour Adelie vs Chinstrap avec `body_mass_g` et `bill_depth_mm`) ne présente **pas de structure claire et bien délimitée en blocs distincts et fortement similaires** pour chaque espèce.

On observe plutôt un mélange de couleurs (similarités) sur l'ensemble de la matrice, y compris dans les blocs censés représenter les similarités intra-classe et inter-classe. Les blocs diagonaux (similarité intra-classe) sont moins uniformément clairs, et les blocs hors diagonale (similarité inter-classe) sont moins uniformément sombres. Il y a un chevauchement des valeurs de similarité entre les observations des deux classes.

Ceci confirme visuellement ce que nous avions anticipé : les espèces Adelie et Chinstrap ne sont pas bien séparables linéairement avec ces deux variables. La matrice de Gram, en montrant un manque de regroupement clair des observations par classe, reflète directement la difficulté du SVM linéaire à trouver une frontière de décision efficace, comme le montrent également le faible score d'accuracy (0.69) et la matrice de confusion précédente.

## 6. SVM à noyau non-linéaire

### Noyau RBF gaussien

Nous allons utiliser un **noyau RBF** ou **radial gaussien**, pour plusieurs valeurs du paramètre gamma. En classe nous avons donné la formule du noyau gaussien :

$k(x, x') = \exp\bigg[-\frac{||x - x'||^2}{2 \sigma^2}\bigg]$

Une autre définition implique le paramètre gamma, $\gamma=\frac{1}{2 \sigma^{2}}$ :

$k(x,x')=\exp\bigg[(-\gamma\||x - x'||^2\bigg]$

Gamma est proportionnel à l'inverse du carré de sigma, qui correspond à *la bande passante* du noyau, ou plus intuitivement le rayon d'influence des observations du train set.
Si **sigma est grand** (donc **gamma petit**) alors les observations du train set vont avoir une influence de longue  portée, et la majorité d'entre eux vont avoir une influence sur la frontière de décision.
Celle ci va donc être "grossière" et lisse (smooth en anglais), quitte à ce que certaines prédictions soient fausses.

Si **sigma est petit** (donc **gamma grand**) alors les observations du train set vont avoir une influence de courte portée, et seules celles proches de la frontière de décision auront une influence localement.
La frontière de décision va donc être "précise" mais on aura tendance à surapprendre.

Vous trouverez une explication claire et détaillée (mais en anglais) ici: https://scikit-learn.org/stable/auto_examples/svm/plot_rbf_parameters.html

### SVM à noyau gaussien (gamma=100)

In [ ]:
# initialisation
model_svc_rbf = svm.SVC(kernel='rbf', C=10, gamma=100)

# entrainement
model_svc_rbf.fit(data_scaled, labels)

Affichons la performance du prédicteur :

In [ ]:
print("Score de la SVM à noyau gaussien (C=10, gamma=100): %.2f" % model_svc_rbf.score(data_scaled, labels))

In [ ]:
metrics.ConfusionMatrixDisplay.from_predictions(labels, model_svc_rbf.predict(data_scaled))

Alternativement :

In [ ]:
metrics.confusion_matrix(labels, model_svc_rbf.predict(data_scaled))

__Question :__ Cette performance correspond-elle à vos attentes ?

**Réponse :**

Oui, cette performance de **0.98 (soit 98%)** correspond tout à fait à nos attentes, et représente une amélioration significative par rapport au SVM linéaire (qui avait un score de 0.69) pour ce problème de classification difficile.

Un noyau non-linéaire comme le noyau RBF gaussien est bien plus adapté pour séparer des classes qui ne sont pas linéairement séparables. L'augmentation de l'accuracy de 0.69 à 0.98 sur les données d'entraînement démontre l'efficacité du noyau RBF à trouver une frontière de décision complexe dans l'espace des caractéristiques transformées.

La matrice de confusion (`array([[148, 3],[1, 67]])`) confirme que le modèle a bien appris à distinguer les deux classes, avec seulement quelques erreurs de classification (3 Adelie mal classés en Chinstrap et 1 Chinstrap mal classé en Adelie), ce qui est excellent pour un problème initialement très difficile.

#### Frontière de décision

Nous pouvons maintenant visualiser la frontière de décision :

In [ ]:
plt.figure(figsize=(8, 6))

# afficher les données (pingouins Adelie)
adelie_indices = np.where(labels==0)[0]
adelie = plt.scatter(data_scaled[adelie_indices][:, 0],
                    data_scaled[adelie_indices][:, 1],
                    label = 'Adelie',
                    cmap=plt.cm.Paired)

# afficher les données (pingouins Chinstrap)
gentoo_indices = np.where(labels==1)[0]
gentoo = plt.scatter(data_scaled[gentoo_indices][:, 0],
                    data_scaled[gentoo_indices][:, 1],
                    label = 'Chinstrap',
                    cmap=plt.cm.Paired)


# Limites du cadre
ax = plt.gca()
xlim = ax.get_xlim()
ylim = ax.get_ylim()

# Marquer les vecteurs de support d'une croix
ax.scatter(model_svc_rbf.support_vectors_[:, 0],
           model_svc_rbf.support_vectors_[:, 1],
           linewidth=1,
           marker='x',
           color='k')

# Grille de points sur lesquels appliquer le modèle
xx = np.linspace(xlim[0], xlim[1], 30)
yy = np.linspace(ylim[0], ylim[1], 30)
YY, XX = np.meshgrid(yy, xx)
xy = np.vstack([XX.ravel(), YY.ravel()]).T
# Prédire pour les points de la grille
Z = model_svc_rbf.decision_function(xy).reshape(XX.shape)

# Afficher la frontière de décision et la marge
ax.contour(XX, YY, Z, colors='k', levels=[-1, 0, 1],
           alpha=0.5, linestyles=['--', '-', '--'])

# Légende
plt.legend()
plt.xlabel("Body mass (centrée-réduite)")
plt.ylabel("Bill depth (centrée-réduite)")
plt.title("Frontière de décision de la SVM à noyau gaussien (C=10, Gamma=100)")
plt.tight_layout()
plt.show()

__Question :__ Que pensez-vous de cette frontière de décision ? Y-a-t'il un risque de surapprentissage ?

**Réponse :**

La frontière de décision est très complexe et sinueuse. Elle s'adapte parfaitement à presque tous les points de données d'entraînement, y compris ceux qui sont un peu isolés ou qui pourraient être du bruit. Cela est typique d'une SVM avec un **grand `gamma`** (ici `gamma=100`). Un grand `gamma` signifie que chaque point de données d'entraînement a une influence de très courte portée, ce qui rend la frontière de décision très sensible aux points individuels.

**Oui, il y a un risque élevé de surapprentissage (overfitting)**. Une frontière de décision aussi complexe et parfaitement ajustée aux données d'entraînement indique que le modèle a probablement appris non seulement le motif général de séparation entre les classes, mais aussi les spécificités et le bruit de l'ensemble d'entraînement. En conséquence, ce modèle risque de ne pas bien se généraliser à de nouvelles données inconnues. Il est probable qu'il aura une excellente performance sur l'ensemble d'entraînement (comme le montre le score de 0.98), mais une performance bien moins bonne sur un ensemble de test indépendant.

#### Matrice de Gram

In [ ]:
GramMatrix = metrics.pairwise.rbf_kernel(data_scaled, gamma=100)
GramMatrix_scaled = center_and_normalise_kernel(GramMatrix)

# heatmap + color map
fig, ax = plt.subplots(figsize=(5, 5))
plot = ax.imshow(GramMatrix_scaled)

# set axes boundaries
ax.set_xlim([0, data.shape[0]]) ; ax.set_ylim([0, data_scaled.shape[0]])

# flip the y-axis
ax.invert_yaxis() ; ax.xaxis.tick_top()

# plot colorbar to the right
plt.colorbar(plot, pad=0.1, fraction=0.04)
plt.show()

__Question :__ Que pensez-vous de cette matrice de Gram ?

**Réponse :**

Avec un `gamma` très élevé (ici `gamma=100`), la matrice de Gram est très différente des précédentes. On observe des valeurs de similarité très localisées.

*   Les éléments sur la diagonale (similarité d'un point avec lui-même) sont bien sûr de 1 (couleur très chaude).
*   Cependant, les valeurs de similarité diminuent **très rapidement** pour les points même légèrement éloignés. La matrice apparaît très 'claire' sur la diagonale et très 'sombre' partout ailleurs, avec peu de nuances intermédiaires.

Cela signifie que chaque point de données n'a d'influence que sur son voisinage immédiat. Les points sont considérés comme similaires uniquement s'ils sont extrêmement proches dans l'espace des caractéristiques. Cette structure de la matrice de Gram est une forte indication de **surapprentissage (overfitting)**. Le modèle s'adapte de manière trop spécifique aux points d'entraînement individuels, et risque de ne pas bien généraliser à de nouvelles données, car de petites variations dans les données de test entraîneraient une similarité très faible.

### Généralisation

Est-ce que ce modèle se __généralise__ bien, autrement dit, sera-t-il capable de faire de bonnes prédictions sur de nouvelles données que nous n'avons pas utilisées pour le construire ?

Pour le savoir, nous allons séparer les données en un __jeu d'entraînement__ et un __jeu de test__. Nous allons entraîner nos SVMs sur le jeu d'entraînement seulement, et mesurer leur performance sur le jeu de test. Le jeu de test, étant inconnu au moment de l'entraînement, fait figure de nouvelles données. Pour cela nous allons utiliser la fonction [train_test_split](http://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) de scikit-learn.

Nous devons faire le split sur les variables non standardisées, standardiser le jeu de train, puis standardiser le jeu de test en fonction de la variance et de la moyenne des variables du jeu de train.

In [ ]:
from sklearn import model_selection

In [ ]:
X_train, X_test, y_train, y_test = model_selection.train_test_split(data,
                                                                    labels,
                                                                    test_size=.2,
                                                                    random_state=21)

In [ ]:
std_scaler = preprocessing.StandardScaler().fit(X_train)
X_train_scaled = std_scaler.transform(X_train)
X_test_scaled = std_scaler.transform(X_test)

Nous allons maintenant calculer l'_accuracy_ d'une SVM sur le jeu d'entraînement et sur le jeu de test pour plusieurs valeurs de `gamma` :

In [ ]:
gamma_values = np.linspace(0.01, 200, 20)

In [ ]:
acc_train, acc_test = list(), list()

for param in gamma_values:
    # Initialisation
    clf = svm.SVC(kernel='rbf', C=10, gamma=param)
    # Entrainement
    clf.fit(X_train_scaled, y_train)
    # Accuracy sur le jeu d'entrainement
    acc_train.append(clf.score(X_train_scaled, y_train))
    # Accuracy sur le jeu de test
    acc_test.append(clf.score(X_test_scaled, y_test))

Représentons la performance en fonction des valeurs de `gamma` testées

In [ ]:
plt.plot(gamma_values, acc_train, label='entrainement', lw=2)
plt.plot(gamma_values, acc_test, label='test', lw=2)

# add a legend
plt.legend(loc='best')

# format the plot
plt.xlabel("Gamma")
plt.ylabel("Accuracy")
plt.tight_layout()

plt.show()

__Question :__ Y-a-t'il surapprentissage ? Pour quelles valeurs de gamma ?

**Réponse :**

Oui, il y a clairement un phénomène de **surapprentissage (overfitting)** observé sur le graphique.

*   Pour les **petites valeurs de gamma** (environ de 0.01 à 20-30), l'accuracy d'entraînement et l'accuracy de test sont relativement proches, et toutes deux assez faibles. Cela suggère un **sous-apprentissage** (underfitting) ou un modèle trop simple qui ne capture pas suffisamment la complexité des données.

*   À partir d'une valeur de gamma d'environ **30-40 et au-delà**, l'accuracy d'entraînement augmente fortement et se stabilise à un niveau très élevé (autour de 0.98), tandis que l'accuracy de test commence à stagner ou même à légèrement diminuer (elle reste aux alentours de 0.60-0.68). Cet écart significatif et croissant entre l'accuracy d'entraînement (élevée) et l'accuracy de test (plus faible et stagnante) est la signature du surapprentissage.

Donc, le surapprentissage commence à se manifester clairement pour des **valeurs de gamma supérieures à environ 30-40**, et devient très prononcé pour les valeurs les plus élevées de gamma testées (jusqu'à 200). Le modèle s'adapte trop spécifiquement aux données d'entraînement et perd sa capacité à généraliser sur de nouvelles données (le jeu de test).

#### Sélection de gamma et C par validation croisée

Nous allons maintenant sélectionner `gamma` et `C` par validation croisée sur le jeu d'entraînement.

Commençons par définir la grille :

In [ ]:
gamma_values = np.logspace(-1, 3, 10)
C_values = np.array([1., 10., 100., 500, 1000])

Nous pouvons maintenant utiliser [GridSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html)  :

In [ ]:
# Instanciation d'un objet GridSearchCV
grid_svc = model_selection.GridSearchCV(svm.SVC(kernel='rbf'), # prédicteur à évaluer
                                       {'gamma': gamma_values, 'C': C_values}, # dictionnaire de valeurs d'hyperparamètres
                                       cv=5, # utiliser 5 folds de validation croisée
                                       scoring='accuracy' # métrique d'évaluation de la performance
                                       )

In [ ]:
%%time

# Utilisation de cet objet sur les données d'entraînement
grid_svc.fit(X_train_scaled, y_train)

La valeur optimale des hyperparamètres est donnée par :

In [ ]:
print(grid_svc.best_params_)

Visualisons l'accuracy en fonction de `C` et `gamma` :

In [ ]:
plt.figure(figsize=(8, 8))

# réarrangement des résultats
scores = grid_svc.cv_results_['mean_test_score'].reshape(len(C_values), len(gamma_values))

# heatmap
plt.imshow(scores, interpolation='none', cmap="RdBu_r")

# colorbar
plt.colorbar()

# Légende
plt.title("Accuracy d'une SVM en validation croisée")
plt.ylabel("C")
plt.xlabel("Gamma")
plt.xlim((-0.5, 3.5))
plt.yticks(np.arange(len(C_values)), C_values)
plt.xticks(np.arange(len(gamma_values)), ["%.2f" % x for x in gamma_values], rotation=90)
plt.tight_layout()
plt.show()

#### Performance de la SVM avec hyperparamètre optimaux sur le jeu de test

In [ ]:
svc_best = grid_svc.best_estimator_

In [ ]:
print("Score sur le jeu de test de la SVM à noyau gaussien (C et gamma optimisés): %.2f" % svc_best.score(X_test_scaled, y_test))

In [ ]:
metrics.ConfusionMatrixDisplay.from_predictions(y_test, svc_best.predict(X_test_scaled))

Alternativement :

In [ ]:
metrics.confusion_matrix(y_test, svc_best.predict(X_test_scaled))

## 7. SVM à noyau non-linéaire sur un problème plus simple

In [ ]:
labels = penguins_labels[penguins_labels["species_int"].isin([0,1])]
labels = np.array(labels["species_int"])
print("shape de y:", labels.shape)

data = penguins_features[penguins_labels["species_int"].isin([0,1])]
data = np.array(data[["body_mass_g", "bill_length_mm"]])
print("shape de X:", data.shape)

In [ ]:
X_train, X_test, y_train, y_test = model_selection.train_test_split(data,
                                                                    labels,
                                                                    test_size=.2,
                                                                    random_state=21)

In [ ]:
std_scaler = preprocessing.StandardScaler().fit(X_train)
X_train_scaled = std_scaler.transform(X_train)
X_test_scaled = std_scaler.transform(X_test)

In [ ]:
plt.figure(figsize=(8, 6))

# afficher les données (pingouins Adelie)
adelie_indices = np.where(y_train==0)[0]
adelie = plt.scatter(X_train_scaled[adelie_indices][:, 0],
                    X_train_scaled[adelie_indices][:, 1],
                    label = 'Adelie',
                    cmap=plt.cm.Paired)

# afficher les données (pingouins Chinstrap)
gentoo_indices = np.where(y_train==1)[0]
gentoo = plt.scatter(X_train_scaled[gentoo_indices][:, 0],
                    X_train_scaled[gentoo_indices][:, 1],
                    label = 'Chinstrap',
                    cmap=plt.cm.Paired)

# Légende
plt.legend()
plt.xlabel("Body mass (centrée-réduite)")
plt.ylabel("Bill length (centrée-réduite)")
plt.title("Données centrées-réduites")
plt.tight_layout()
plt.show()

In [ ]:
gamma_values = np.logspace(-1, 3, 10)
C_values = np.array([1., 10., 100., 500, 1000])

Nous pouvons maintenant utiliser [GridSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html)  :

In [ ]:
# Instanciation d'un objet GridSearchCV
grid_svc = model_selection.GridSearchCV(svm.SVC(kernel='rbf'), # prédicteur à évaluer
                                       {'gamma': gamma_values, 'C': C_values}, # dictionnaire de valeurs d'hyperparamètres
                                       cv=5, # utiliser 5 folds de validation croisée
                                       scoring='accuracy' # métrique d'évaluation de la performance
                                       )

In [ ]:
%%time

# Utilisation de cet objet sur les données d'entraînement
grid_svc.fit(X_train_scaled, y_train)

La valeur optimale des hyperparamètres est donnée par :

In [ ]:
print(grid_svc.best_params_)

In [ ]:
# standardisation (centrer-réduire)
std_scaler = preprocessing.StandardScaler().fit(data)
data_scaled = std_scaler.transform(data)


GramMatrix = metrics.pairwise.rbf_kernel(data_scaled, gamma=grid_svc.best_params_['gamma'])
GramMatrix_scaled = center_and_normalise_kernel(GramMatrix)

# heatmap + color map
fig, ax = plt.subplots(figsize=(5, 5))
plot = ax.imshow(GramMatrix_scaled)

# set axes boundaries
ax.set_xlim([0, data.shape[0]]) ; ax.set_ylim([0, data_scaled.shape[0]])

# flip the y-axis
ax.invert_yaxis() ; ax.xaxis.tick_top()

# plot colorbar to the right
plt.colorbar(plot, pad=0.1, fraction=0.04)
plt.show()

Visualisons l'accuracy en fonction de `C` et `gamma` :

In [ ]:
plt.figure(figsize=(8, 8))

# réarrangement des résultats
scores = grid_svc.cv_results_['mean_test_score'].reshape(len(C_values), len(gamma_values))

# heatmap
plt.imshow(scores, interpolation='none', cmap="RdBu_r")

# colorbar
plt.colorbar()

# Légende
plt.title("Accuracy d'une SVM en validation croisée")
plt.ylabel("C")
plt.xlabel("Gamma")
plt.xlim((-0.5, 3.5))
plt.yticks(np.arange(len(C_values)), C_values)
plt.xticks(np.arange(len(gamma_values)), ["%.2f" % x for x in gamma_values], rotation=90)
plt.tight_layout()
plt.show()

In [ ]:
svc_best = grid_svc.best_estimator_

In [ ]:
plt.figure(figsize=(8, 6))

# afficher les données (pingouins Adelie)
adelie_indices = np.where(y_train==0)[0]
adelie = plt.scatter(X_train_scaled[adelie_indices][:, 0],
                    X_train_scaled[adelie_indices][:, 1],
                    label = 'Adelie',
                    cmap=plt.cm.Paired)

# afficher les données (pingouins Chinstrap)
gentoo_indices = np.where(y_train==1)[0]
gentoo = plt.scatter(X_train_scaled[gentoo_indices][:, 0],
                    X_train_scaled[gentoo_indices][:, 1],
                    label = 'Chinstrap',
                    cmap=plt.cm.Paired)


# Limites du cadre
ax = plt.gca()
xlim = ax.get_xlim()
ylim = ax.get_ylim()

# Marquer les vecteurs de support d'une croix
ax.scatter(svc_best.support_vectors_[:, 0],
           svc_best.support_vectors_[:, 1],
           linewidth=1,
           marker='x',
           color='k')

# Grille de points sur lesquels appliquer le modèle
xx = np.linspace(xlim[0], xlim[1], 30)
yy = np.linspace(ylim[0], ylim[1], 30)
YY, XX = np.meshgrid(yy, xx)
xy = np.vstack([XX.ravel(), YY.ravel()]).T
# Prédire pour les points de la grille
Z = svc_best.decision_function(xy).reshape(XX.shape)

# Afficher la frontière de décision et la marge
ax.contour(XX, YY, Z, colors='k', levels=[-1, 0, 1],
           alpha=0.5, linestyles=['--', '-', '--'])

# Légende
plt.legend()
plt.xlabel("Body mass (centrée-réduite)")
plt.ylabel("Bill length (centrée-réduite)")
plt.title(f"Frontière de décision de la SVM à noyau gaussien (C={svc_best.C}, Gamma={svc_best.gamma: .2f})")
plt.tight_layout()
plt.show()

In [ ]:
print("Score sur le jeu de test de la SVM à noyau gaussien (C et gamma optimisés): %.2f" % svc_best.score(X_test_scaled, y_test))

In [ ]:
metrics.ConfusionMatrixDisplay.from_predictions(y_test, svc_best.predict(X_test_scaled))

Alternativement :

In [ ]:
metrics.confusion_matrix(y_test, svc_best.predict(X_test_scaled))

## Supplément : Trouver le meilleur couple de variables

On peut utiliser le package `seaborn` pour examiner aisément les paires de variables deux à deux et en déduire lesquelles sont les plus utiles pour séparer les classes :

In [ ]:
penguins_adelie_chinstrap = pd.concat([penguins_features[penguins_labels["species_int"].isin([0,1])],
                                    penguins_labels[penguins_labels["species_int"].isin([0,1])]],
                                     axis = 1)

In [ ]:
import seaborn as sns
sns.pairplot(penguins_adelie_chinstrap, hue="species_int",palette="bright")

## Conclusion

Nous sommes arrivés à la fin de ce notebook. Voici un résumé de ce que nous avons couvert, avec les points clés :

- Nous avons utilisé la bibliothèque `scikit-learn` pour classifier les manchots (trois espèces : Adelie, Gentoo et Chinstrap) à partir de leurs caractéristiques physiques (longueur/profondeur du bec, masse corporelle, longueur des palmes).
- Nous avons commencé par les **SVM linéaires** ([SVC](https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html) avec `kernel='linear'`), qui trouvent un hyperplan séparateur de marge maximale. L'hyperparamètre `C` contrôle le compromis entre la marge et les erreurs de classification : plus `C` est grand, moins il y a de tolérance aux violations de la marge.
- Nous avons visualisé les **vecteurs de support** (points proches de la frontière de décision) et compris comment ils définissent la frontière. Nous avons également découvert la **matrice de Gram** (ou noyau linéaire), qui représente les similarités entre observations et permet de visualiser à quel point les classes sont séparables.
- Nous avons utilisé la **validation croisée** pour évaluer la généralisation, et la **recherche en grille** ([GridSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html)) pour optimiser les hyperparamètres en testant systématiquement des combinaisons de valeurs.
- Pour les problèmes **non-linéairement séparables**, nous avons exploré les **SVM à noyaux non-linéaires**, en particulier le **noyau RBF gaussien** ([`kernel='rbf'`](https://scikit-learn.org/stable/modules/svm.html#rbf-kernel)). Ce noyau projette implicitement les données dans un espace de dimension supérieure, permettant de séparer les classes non-linéaires.
- L'hyperparamètre **gamma** contrôle la bande passante du noyau RBF : gamma petit (sigma grand) produit une frontière lisse mais peut sous-apprendre; gamma grand (sigma petit) produit une frontière précise mais peut surapprendre. Nous avons observé ce compromis en traçant l'accuracy en entraînement vs test selon gamma.
- Nous avons montré l'importance du **prétraitement** (centrer-réduire les données) pour que les SVMs exploitent correctement la distance euclidienne, en particulier quand les échelles des variables diffèrent.
- Enfin, nous avons utilisé `GridSearchCV` pour co-optimiser `C` et `gamma` par validation croisée, assurant une meilleure généralisation aux données non vues.
